# 21 · E1h: the same comparison, done consistently. Every model starts from the true resting state

**What went wrong in Notebook 20.** Its own sanity check failed: **EXPRESSIBLE = False**.
* The rescaled L1 and the frozen-gate slow current, which are algebraically the same equation, differed by up to **0.52**.
* The registered verdicts were therefore not interpretable. That run had the rescaled L1 *losing* to the fitted L1 (−0.015 at the smallest step, ρ = 0.05), while the slow current gained +0.040 over it.

**Diagnosis (reproduced on a synthetic worm while building this).**
* The equivalence is exact **for the same starting state**, and the two did not start from the same state:
  * `simulate_numpy` runs every model from zero for a 30-s burn-in;
  * the slow-current runs start from L1's state *after* that burn-in, and freeze their gate there.
* That only matters if 30 s is not enough to reach rest, and for the fitted L1s it probably is not. Notebook 05 measured their largest Lyapunov exponents at −0.08 to −0.16 /s, i.e. a slowest relaxation of ~6–12 s, so ~9 % of the initial transient is left after 30 s.
* A synthetic L1 slowed to the same relaxation rate (−0.064 /s) reproduced the failure:

| synthetic L1, slow relaxation | max difference rescaled L1 vs frozen gate |
|---|---|
| from different starts (as in Notebook 20) | 2.5 × 10⁻² |
| both from the exact fixed point | 5 × 10⁻¹¹ |

* Even plain L1's predictions moved by 0.59 × noise between "from zero, 30-s burn-in" and "from the exact fixed point".

**What this puts in doubt.**
1. Notebook 20's verdicts.
2. **Notebook 19's +0.025 gain.** Its slow currents froze their gate at a state that was not yet rest, which acts as a constant extra input that the gate then slowly releases. Part of the gain could come from that, not from adaptation.
3. **E1 itself.** The fitted L1 was trained and scored with a 30-s burn-in from zero. If the network had not reached rest, its predictions include a leftover transient that the fit may have used.

**This notebook redoes Notebooks 19–20 consistently.**
* For every fold model, the exact resting state is solved (`intrinsic.true_rest`: root finding plus a stability check).
* **Every** model then starts there: L1\*, the rescaled L1 (A), and the slow current (B).
* It also measures how much E1's own predictions depend on the unfinished burn-in.

In [1]:
#@title Setup — run first (works locally on a Mac/PC and on Colab)
import os, sys, subprocess
REPO_URL   = "https://github.com/aslansd/Closure-Bench.git"  # Colab only: clone from here
DRIVE_PATH = "/content/drive/MyDrive/Closure-Bench"          # Colab only: ...or use a Drive copy
IN_COLAB = "google.colab" in sys.modules

def _find_package():
    # 1) explicit location, 2) the repo this notebook lives in, 3) Colab clone / Drive
    cands = [os.environ.get("CLOSUREBENCH_ROOT", ""), ".", "..", "/content/Closure-Bench", DRIVE_PATH]
    for root in cands:
        if root and os.path.isdir(os.path.join(root, "closurebench")):
            return os.path.abspath(root)
    if IN_COLAB:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "wormneuroatlas", "optax"], check=False)
        if subprocess.run(["git", "clone", "-q", REPO_URL, "/content/Closure-Bench"]).returncode == 0:
            return "/content/Closure-Bench"
        from google.colab import drive
        drive.mount("/content/drive")
        if os.path.isdir(os.path.join(DRIVE_PATH, "closurebench")):
            return DRIVE_PATH
    raise RuntimeError("closurebench not found. Locally: open this notebook from Closure-Bench/notebooks/, "
                       "or set the environment variable CLOSUREBENCH_ROOT to the Closure-Bench folder.")

ROOT = _find_package()
sys.path.insert(0, ROOT)
try:
    import jax, optax, wormneuroatlas  # noqa: F401
except ImportError as e:
    raise ImportError(f"{e}. Locally, create the environment first: see README.md "
                      "('Run locally'), then select the 'Python (closurebench)' kernel.") from None
RESULTS = os.environ.get("CLOSUREBENCH_RESULTS", os.path.join(ROOT, "results"))
os.makedirs(RESULTS, exist_ok=True)
from closurebench import config as CBC
MODE = CBC.get_mode()        # 'smoke' | 'laptop' | 'full'  (env CLOSUREBENCH_MODE, default: full on GPU, laptop on CPU)
print("closurebench:", ROOT); print("results:", RESULTS); print(CBC.describe_machine()); print("MODE:", MODE)

closurebench: /Users/asataryd/Documents/ENI-Projects/Aslan/NeuroAILab/Githubs/Closure-Bench-Mac
results: /Users/asataryd/Documents/ENI-Projects/Aslan/NeuroAILab/Githubs/Closure-Bench-Mac/results
Darwin arm64 | python 3.11.5 | jax 0.10.2 on cpu (cpu) | CPU cores: 8
MODE: laptop


In [2]:
import json, glob, pickle, time, datetime, numpy as np, matplotlib.pyplot as plt, jax
import closurebench
assert tuple(int(x) for x in closurebench.__version__.split(".")[:3]) >= (0, 18, 0), \
    f"Notebook 21 needs closurebench >= 0.18.0 (found {closurebench.__version__}); copy the full package and restart the kernel"
from closurebench import data as D, ladder as lad, metrics as M_, substrates as SB, conductance as CD, intrinsic as IN
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
CONF = {"smoke":  dict(N_COLS=10, FOLDS_USED=1),
        "laptop": dict(N_COLS=None, FOLDS_USED=None),
        "full":   dict(N_COLS=None, FOLDS_USED=None)}[MODE]
globals().update(CONF)
RHOS, TAU_NS = (0.0, 0.1, 0.2, 0.3, 0.45, 0.6), (0.5, 1.0, 2.0, 5.0, 10.0, 20.0)   # Notebook 19's grid (finite tau_n)
RHO_A = (0.0, 0.05, 0.1, 0.15, 0.2, 0.3, 0.45)                                        # rescaled L1 (tau_n = infinity)
N_BOOT, E1_THRESHOLD = 2000, 0.05
TAG = "" if MODE == "full" else f"_{MODE}"

# ---- the fitted L1 models of Notebook 04 (as in Notebook 05); smoke falls back to a synthetic L1 if they are absent
E1_TAG = "_laptop" if MODE == "smoke" else TAG
try:
    e1_result = json.load(open(os.path.join(RESULTS, f"E1_result{E1_TAG}.json")))
    e1_plan = json.load(open(os.path.join(RESULTS, e1_result["preregistration"])))
    assert e1_result["L_star"] == "L1", "conductance.py implements L1"
    c = e1_plan["config"]
    ds = D.Dataset.load(os.path.join(RESULTS, "atlas_dataset.npz"))
    if c["n_neurons"] < ds.N:
        mk = ds.mask()[0]; score = mk.sum(1).astype(float); score[ds.stim] += mk.sum(0)
        ds = D.subset(ds, np.sort(np.argsort(-score)[:c["n_neurons"]]))
    st = lad.Structure.from_dataset(ds, tie=c["tie_classes"]); cfg = lad.SimConfig(**c["sim"])
    folds = M_.kfold_pairs(ds, c["k_folds"], c["fold_seed"])
    ck1 = os.path.join(RESULTS, f"e1_checkpoints{E1_TAG}")
    models = [pickle.load(open(os.path.join(ck1, f"fold{f}_L1_final.pkl"), "rb"))["params"] for f in range(c["k_folds"])]
    SOURCE = f"E1 fits ({E1_TAG.strip('_') or 'full'})"
except (FileNotFoundError, KeyError) as err:
    if MODE != "smoke":
        raise
    print("E1 fits not found (", err, ") -> smoke test on a synthetic L1")
    ds, p_true, _ = lad.make_synthetic(lad.random_wiring(N=30, S=10, seed=1, strains=("wt",)), true_level="L1", seed=1)
    st = lad.Structure.from_dataset(ds); cfg = lad.SimConfig(); folds = [None]; models = [p_true]; SOURCE = "synthetic L1"
rng = np.random.default_rng(0)
COLS = np.sort(rng.choice(ds.S, N_COLS, replace=False)) if N_COLS and N_COLS < ds.S else np.arange(ds.S)
FOLDS = list(range(len(models)))[:FOLDS_USED] if FOLDS_USED else list(range(len(models)))
print(f"source: {SOURCE} | {ds.N} neurons | {len(COLS)} stimuli | fold models {FOLDS}")
if folds[0] is None:                       # smoke fallback: a synthetic L1 has no folds -> make two (negative control: no adaptation in the truth)
    folds = M_.kfold_pairs(ds, 2, 0); models = [models[0]] * 2; FOLDS = [0, 1]
print(f"every model starts from the exact resting state | A: rescaled L1, rho {RHO_A} | B: A plus slow current rho {RHOS[1:]} x tau_n {TAU_NS} s")

source: E1 fits (laptop) | 120 neurons | 120 stimuli | fold models [0, 1, 2, 3, 4]
every model starts from the exact resting state | A: rescaled L1, rho (0.0, 0.05, 0.1, 0.15, 0.2, 0.3, 0.45) | B: A plus slow current rho (0.1, 0.2, 0.3, 0.45, 0.6) x tau_n (0.5, 1.0, 2.0, 5.0, 10.0, 20.0) s


### Pre-registration (before §1)

Recorded in `results/E1h_preregistration*.json`. This amends the registrations of Notebooks 19 and 20; their selection, scoring and bootstrap procedures are unchanged. **L1\*** is the fitted L1 started at its exact resting state. If the solver fails or the fixed point is unstable for a fold, that fold uses a 300-s burn-in instead, and this is reported.

* **EXPRESSIBLE\*** (must hold, else stop): the rescaled L1 from the fixed point equals the frozen-gate slow current from the fixed point to < 10⁻⁶ (fold 0).
* **PROTOCOL_DEPENDENT** (reported): on some fold, L1 under E1's protocol (from zero, 30-s burn-in) differs from L1\* by gap/noise ≥ 0.1 or |held-out ΔFEVE| ≥ 0.05.
* **SLOW_GAIN_SURVIVES** (Notebook 19 re-tested): B beats L1\* on held-out pairs, CI lower bound > 0.
* **UNDERFIT_L1** (Notebook 20 re-tested): A beats L1\*, CI lower bound > 0.
* **ADAPTATION_DYNAMICS** (Notebook 20 re-tested): B beats A, CI lower bound > 0.
* **VALID:** on L1\*-generated null data, neither A nor B beats L1\* (CI lower bounds ≤ 0).

In [3]:
prereg = os.path.join(RESULTS, f"E1h_preregistration{TAG}.json")
plan = {"experiment": "E1h consistent restart from the exact resting state", "mode": MODE, "registered_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
        "source": SOURCE, "amends": ["E1f (Notebook 19)", "E1g (Notebook 20)"],
        "config": {"folds": FOLDS, "rho_A": list(RHO_A), "rho_B": list(RHOS), "tau_n_B": list(TAU_NS), "n_boot": N_BOOT,
                   "start": "exact fixed point (intrinsic.true_rest); fallback 300-s burn-in if not ok"},
        "EXPRESSIBLE*": "rescaled L1 == frozen-gate slow current, both from the fixed point, < 1e-6 (fold 0)",
        "PROTOCOL_DEPENDENT": "L1 (zero + 30-s burn-in) vs L1* gap/noise >= 0.1 or |held-out dFEVE| >= 0.05 on some fold",
        "SLOW_GAIN_SURVIVES": "B beats L1*, CI lower bound > 0", "UNDERFIT_L1": "A beats L1*, CI lower bound > 0",
        "ADAPTATION_DYNAMICS": "B beats A, CI lower bound > 0", "VALID": "on L1*-generated null data neither A nor B beats L1* (CI lower <= 0)"}
if os.path.exists(prereg):
    plan = json.load(open(prereg)); print("Existing registration (", plan["registered_at"], ") used unchanged.")
else:
    json.dump(plan, open(prereg, "w"), indent=2); print("Registered ->", prereg)

Registered -> /Users/asataryd/Documents/ENI-Projects/Aslan/NeuroAILab/Githubs/Closure-Bench-Mac/results/E1h_preregistration_laptop.json


## 1 · The exact resting state of every fold model, and every simulation from it (checkpointed)

In [4]:
import dataclasses
CK = os.path.join(RESULTS, "e1h_ck", MODE); os.makedirs(CK, exist_ok=True)
SIM, DIAG = {}, {}
for f in FOLDS:
    t0 = time.time()
    p = jax.tree_util.tree_map(jax.numpy.asarray, models[f])
    e = SB.effective_params("L1", p, st, COLS)
    V30, _ = CD.rest_state(e, cfg)
    r = IN.true_rest(e, V30)
    if r["ok"]:
        Vs, how = r["V"], "exact fixed point"
    else:
        Vs, _ = CD.rest_state(e, dataclasses.replace(cfg, t_burn=300.0)); how = "300-s burn-in (fixed point not ok)"
    DIAG[f] = dict(residual_30s=IN.fixed_point_residual(e, V30), distance_30s=float(np.abs(V30 - Vs).max()),
                   max_real_eig=r["max_real_eig"], start=how)
    print(f"fold {f}: start = {how} | after E1's 30-s burn-in: residual {DIAG[f]['residual_30s']:.2e}, max distance to rest {DIAG[f]['distance_30s']:.3f} | "
          f"slowest relaxation {r['max_real_eig']:+.3f} /s")
    ck = os.path.join(CK, f"fold{f}.npz")
    if os.path.exists(ck):
        z = np.load(ck); SIM[f] = {k: z[k] for k in z.files}
    else:
        S = {"protocol": SB.simulate_numpy(e, ds.stim[COLS], cfg, "rk4", dt=0.02), "A_0.0": IN.simulate_l1_from(e, ds.stim[COLS], cfg, Vs)}
        for rho in RHO_A[1:]:
            S[f"A_{rho}"] = IN.simulate_l1_from(IN.rescaled_l1(e, Vs, rho), ds.stim[COLS], cfg, Vs)
        for rho in RHOS[1:]:
            for tn in TAU_NS:
                S[f"B_{rho}_{tn}"] = IN.simulate_slow_k(e, ds.stim[COLS], cfg, Vs, rho=rho, tau_n=tn)
        if f == FOLDS[0]:
            S["check_frozen_0.1"] = IN.simulate_slow_k(e, ds.stim[COLS], cfg, Vs, rho=0.1, tau_n=1e9)
        np.savez(ck, **S); SIM[f] = S
    if f == FOLDS[0]:
        dmax = float(np.abs(SIM[f]["check_frozen_0.1"] - SIM[f]["A_0.1"]).max())
        EXPRESSIBLE = dmax < 1e-6
        print(f"  EXPRESSIBLE* check: max |rescaled L1 - frozen slow current| from the fixed point = {dmax:.2e} -> {EXPRESSIBLE}")
    print(f"  {len(SIM[f])} simulations ({time.time() - t0:.0f} s)", flush=True)
assert EXPRESSIBLE, "EXPRESSIBLE* failed: the comparison below would not be interpretable"

fold 0: start = exact fixed point | after E1's 30-s burn-in: residual 3.73e-03, max distance to rest 0.044 | slowest relaxation -0.085 /s
  EXPRESSIBLE* check: max |rescaled L1 - frozen slow current| from the fixed point = 3.47e-09 -> True
  39 simulations (99 s)
fold 1: start = exact fixed point | after E1's 30-s burn-in: residual 4.49e-03, max distance to rest 0.054 | slowest relaxation -0.079 /s
  38 simulations (109 s)
fold 2: start = exact fixed point | after E1's 30-s burn-in: residual 2.62e-03, max distance to rest 0.024 | slowest relaxation -0.105 /s
  38 simulations (129 s)
fold 3: start = exact fixed point | after E1's 30-s burn-in: residual 3.38e-03, max distance to rest 0.038 | slowest relaxation -0.085 /s
  38 simulations (108 s)
fold 4: start = exact fixed point | after E1's 30-s burn-in: residual 3.43e-03, max distance to rest 0.034 | slowest relaxation -0.093 /s
  38 simulations (99 s)


## 2 · How much do E1's predictions depend on the unfinished burn-in? (PROTOCOL_DEPENDENT)

In [5]:
import copy
def full(R):
    out = np.zeros(ds.mean.shape[:3]); out[:, :, COLS] = R[None]; return out
def split(f):
    test = folds[f]; train = np.zeros_like(test)
    for g in range(len(folds)):
        if g != f: train |= folds[g]
    return train, test
PROTO = []
for f in FOLDS:
    train, testm = split(f)
    gp = SB.substrate_gap(SIM[f]["protocol"], SIM[f]["A_0.0"], ds, COLS)
    dfe = M_.feve(full(SIM[f]["A_0.0"]), ds, strain="wt", extra_mask=testm) - M_.feve(full(SIM[f]["protocol"]), ds, strain="wt", extra_mask=testm)
    PROTO.append((gp, dfe))
    print(f"fold {f}: L1 from zero (E1 protocol) vs L1* from rest: gap/noise {gp:.3f}, held-out FEVE(L1*) - FEVE(protocol) {dfe:+.4f}")
PROTOCOL_DEPENDENT = any(g >= 0.1 or abs(d) >= 0.05 for g, d in PROTO)
print("Registered (reported) -> PROTOCOL_DEPENDENT:", PROTOCOL_DEPENDENT)

fold 0: L1 from zero (E1 protocol) vs L1* from rest: gap/noise 0.001, held-out FEVE(L1*) - FEVE(protocol) -0.0016
fold 1: L1 from zero (E1 protocol) vs L1* from rest: gap/noise 0.000, held-out FEVE(L1*) - FEVE(protocol) -0.0057
fold 2: L1 from zero (E1 protocol) vs L1* from rest: gap/noise 0.000, held-out FEVE(L1*) - FEVE(protocol) -0.0003
fold 3: L1 from zero (E1 protocol) vs L1* from rest: gap/noise 0.000, held-out FEVE(L1*) - FEVE(protocol) -0.0017
fold 4: L1 from zero (E1 protocol) vs L1* from rest: gap/noise 0.000, held-out FEVE(L1*) - FEVE(protocol) -0.0024
Registered (reported) -> PROTOCOL_DEPENDENT: False


## 3 · Notebooks 19–20 redone from the resting state

In [6]:
def run(data):
    cv = {k: np.zeros(data.mean.shape[:3]) for k in ("protocol", "L1s", "A", "B")}; ch = {}
    for f in FOLDS:
        train, test = split(f)
        candA = {k: v for k, v in SIM[f].items() if k.startswith("A_")}
        candB = {**candA, **{k: v for k, v in SIM[f].items() if k.startswith("B_")}}
        tr = {k: M_.feve(full(R), data, strain="wt", extra_mask=train) for k, R in candB.items()}
        a = max(candA, key=tr.get); b = max(candB, key=tr.get)
        for name, R in (("protocol", SIM[f]["protocol"]), ("L1s", SIM[f]["A_0.0"]), ("A", candA[a]), ("B", candB[b])):
            cv[name][test] = full(R)[test]
        ch[f] = (a, b)
    return cv, ch
def test(cv, data, x, y):
    point, boots = M_.bootstrap_feve({x: cv[x], y: cv[y]}, data, [x, y], n_boot=N_BOOT, seed=1, strain="wt")
    return point[x] - point[y], np.percentile(boots[x] - boots[y], [2.5, 97.5]), point
cv, ch = run(ds)
for f, (a, b) in ch.items():
    print(f"fold {f}: A chooses {a} | B chooses {b}")
res = {}
for x, y in (("L1s", "protocol"), ("B", "L1s"), ("A", "L1s"), ("B", "A"), ("B", "protocol")):
    g, ci, pt = test(cv, ds, x, y); res[f"{x}-{y}"] = (g, ci)
    print(f"{x:>8s} - {y:<8s}: {g:+.4f}  95% CI [{ci[0]:+.4f}, {ci[1]:+.4f}]")
_, _, pt = test(cv, ds, "L1s", "protocol"); _, _, pt2 = test(cv, ds, "A", "B")
print(f"\nheld-out FEVE (WT): L1 protocol {pt['protocol']:.4f} | L1* {pt['L1s']:.4f} | A {pt2['A']:.4f} | B {pt2['B']:.4f}")

fold 0: A chooses A_0.05 | B chooses B_0.1_20.0
fold 1: A chooses A_0.05 | B chooses B_0.1_10.0
fold 2: A chooses A_0.05 | B chooses B_0.1_20.0
fold 3: A chooses A_0.05 | B chooses B_0.2_5.0
fold 4: A chooses A_0.05 | B chooses B_0.1_20.0
     L1s - protocol: -0.0024  95% CI [-0.0044, -0.0005]
       B - L1s     : +0.0250  95% CI [+0.0105, +0.0408]
       A - L1s     : +0.0125  95% CI [-0.0008, +0.0260]
       B - A       : +0.0125  95% CI [+0.0042, +0.0235]
       B - protocol: +0.0225  95% CI [+0.0091, +0.0373]

held-out FEVE (WT): L1 protocol 0.2630 | L1* 0.2606 | A 0.2731 | B 0.2856


## 4 · Null control and verdicts

In [7]:
rng = np.random.default_rng(7)
ds_null = copy.deepcopy(ds)
for f in FOLDS:
    m = folds[f] & ds.mask(); ds_null.mean[m] = full(SIM[f]["A_0.0"])[m]
m = ds.mask(); ds_null.mean[m] = ds_null.mean[m] + rng.normal(size=m.sum()) * np.sqrt(ds.var_mean[m])
cv0, ch0 = run(ds_null)
gA0, ciA0, _ = test(cv0, ds_null, "A", "L1s"); gB0, ciB0, _ = test(cv0, ds_null, "B", "L1s")
print("null-control choices:", ch0)
print(f"null: A - L1* {gA0:+.4f} [{ciA0[0]:+.4f}, {ciA0[1]:+.4f}] | B - L1* {gB0:+.4f} [{ciB0[0]:+.4f}, {ciB0[1]:+.4f}]")
VALID = bool(ciA0[0] <= 0 and ciB0[0] <= 0)
SLOW_GAIN_SURVIVES = bool(res["B-L1s"][1][0] > 0)
UNDERFIT_L1 = bool(res["A-L1s"][1][0] > 0)
ADAPTATION_DYNAMICS = bool(res["B-A"][1][0] > 0)
print(f"\nRegistered rules -> EXPRESSIBLE*: {EXPRESSIBLE} | VALID: {VALID} | PROTOCOL_DEPENDENT: {PROTOCOL_DEPENDENT} | "
      f"SLOW_GAIN_SURVIVES: {SLOW_GAIN_SURVIVES} | UNDERFIT_L1: {UNDERFIT_L1} | ADAPTATION_DYNAMICS: {ADAPTATION_DYNAMICS}")

null-control choices: {0: ('A_0.0', 'B_0.2_0.5'), 1: ('A_0.0', 'B_0.3_0.5'), 2: ('A_0.0', 'B_0.3_0.5'), 3: ('A_0.0', 'B_0.3_0.5'), 4: ('A_0.0', 'B_0.45_0.5')}
null: A - L1* +0.0000 [+0.0000, +0.0000] | B - L1* -0.0124 [-0.0353, -0.0016]

Registered rules -> EXPRESSIBLE*: True | VALID: True | PROTOCOL_DEPENDENT: False | SLOW_GAIN_SURVIVES: True | UNDERFIT_L1: False | ADAPTATION_DYNAMICS: True


In [8]:
out = {"source": SOURCE, "diagnostics": {str(f): d for f, d in DIAG.items()}, "protocol": [list(x) for x in PROTO],
       "EXPRESSIBLE": EXPRESSIBLE, "VALID": VALID, "PROTOCOL_DEPENDENT": PROTOCOL_DEPENDENT, "SLOW_GAIN_SURVIVES": SLOW_GAIN_SURVIVES,
       "UNDERFIT_L1": UNDERFIT_L1, "ADAPTATION_DYNAMICS": ADAPTATION_DYNAMICS,
       "tests": {k: {"point": g, "ci": list(ci)} for k, (g, ci) in res.items()},
       "null": {"A": [gA0, list(ciA0)], "B": [gB0, list(ciB0)]}, "choices": {str(f): list(v) for f, v in ch.items()}}
json.dump(out, open(os.path.join(RESULTS, f"E1h_result{TAG}.json"), "w"), indent=2, default=float)
print("saved", f"E1h_result{TAG}.json")

saved E1h_result_laptop.json


## Results (laptop mode, executed on the M1: all 5 fold models, all 120 stimuli)

**Resting states.** The exact fixed point was found and stable for every fold. The slowest relaxation is −0.079 to −0.105 /s, i.e. 10–13 s, consistent with Notebook 05's Lyapunov exponents. After E1's 30-s burn-in the network is still 0.024–0.054 (max over neurons) from rest (residual 3 × 10⁻³).

**EXPRESSIBLE\*: yes.** From the fixed point, the rescaled L1 and the frozen-gate slow current agree to 3.5 × 10⁻⁹. The diagnosis of Notebook 20 is confirmed, and the comparisons below are interpretable.

**PROTOCOL_DEPENDENT: no.** Starting from zero with a 30-s burn-in (E1's protocol) instead of from the exact rest changes L1's predictions by gap/noise ≤ 0.001, with held-out ΔFEVE −0.0003 to −0.006 per fold (pooled −0.0024 [−0.0044, −0.0005]). The leftover transient is real but negligible: **E1's results and the substrate tests of Notebooks 05 and 16–18 stand.**

**Notebooks 19–20, redone consistently (cross-validated held-out FEVE, WT):**

| model | held-out FEVE |
|---|---|
| L1 (E1 protocol) | 0.2630 |
| L1\* (exact rest) | 0.2606 |
| rescaled L1 (A; ρ = 0.05 on every fold) | 0.2731 |
| slow current (B; ρ = 0.1 / τ_n 20 s on 3 folds, 0.1 / 10 s and 0.2 / 5 s on one each) | **0.2856** |

| comparison | gain | 95 % CI | registered rule |
|---|---|---|---|
| B − L1\* | **+0.025** | [+0.011, +0.041] | **SLOW_GAIN_SURVIVES: yes** |
| A − L1\* | +0.0125 | [−0.001, +0.026] | UNDERFIT_L1: no |
| B − A | **+0.0125** | [+0.004, +0.024] | **ADAPTATION_DYNAMICS: yes** |
| B − L1 (E1 protocol) | +0.023 | [+0.009, +0.037] | — |
| null control: A / B − L1\* | 0.000 / −0.012 | upper bounds ≤ 0 | VALID: yes |

**Outcome: the third row of the table below.**
* Notebook 19's gain survives a consistent start, and is not an artefact of the frozen gate.
* About half of it a rescaled L1 also gets (not significant on its own); the other half needs finite-τ_n slow dynamics.
* The total (+0.025) remains **half of E1's 0.05 sufficiency margin**.

**A caution on "beyond any L1".** ADAPTATION_DYNAMICS compares the slow current with *one direction* inside the L1 family (uniform rescaling). It does not show that no L1 could do as well. Nothing here was fitted by gradient: the slow current was added to a fixed L1. **Notebook 22 (E1i) fits both families with the same optimizer, budget and start.**

> **Follow-up (Notebook 22).** Fitted with the same optimizer and budget, the slow-current rung `L1s` beats a re-fitted L1 by +0.024 [+0.014, +0.039]. Its share of the leak grows to 0.15–0.17 on every fold, while its timescale is not identifiable. From "nearly off" (E1's warm start) it barely grows.

## 5 · Reading the result

| SLOW_GAIN_SURVIVES | UNDERFIT_L1 | ADAPTATION_DYNAMICS | reading |
|---|---|---|---|
| no | – | – | Notebook 19's gain came from the inconsistent start (a gate frozen before rest). There is no evidence for slow adaptation, and E1's bracket stands. |
| yes | yes | no | The gain is a better point inside the L1 family: E1's L1 was under-fitted. |
| **yes ← observed** | **no ← observed** | **yes ← observed** | Slow dynamics beyond the rescaled L1 (one direction of the L1 family; see Results) improve held-out predictions: the first positive evidence for a rung deeper than L1. Still, if the gain is < 0.05, it is below E1's sufficiency margin. |

**PROTOCOL_DEPENDENT** (observed: **no**), if true, would have been a finding about E1 itself:
* the fitted models' predictions depend on a 30-s burn-in that does not reach rest;
* E1 (and every substrate test in Notebooks 05 and 16–18, which reproduced the same protocol) evaluated "the fitted program" together with a leftover transient;
* a re-fit of E1 should start every simulation from the exact resting state, or use a burn-in of several relaxation times.

**Limits.**
* The exact resting state is solved for the *fitted* parameters. A model refitted under the corrected protocol could rest elsewhere.
* Everything else is as in Notebooks 19–20.